# Geração dos layouts

As duas modelagens geram os layouts a partir de L0, o layout aleatório de referência:

- **L1 (grafo multicamadas):** busca local que minimiza $F(\kappa) = \sum_{\{s_i,s_j\} \in E_2} \omega_{ij}\, D(\kappa(s_i), \kappa(s_j))$, com $\omega_{ij} = \text{lift} - 1$.
- **L2 (grafo único):** as seções mais pedidas no prior ocupam os slots de menor $\rho(p) = d(\text{ENTRADA}, p) + d(p, \text{CAIXA})$.

O valor $n_{min} = 300$ vem da calibração feita na modelagem (`dados/amostras/calibracao_n_min.csv`), refeita no fim deste notebook. L0 e as amostras de pedidos também vieram da modelagem e são lidos dos arquivos em `dados/`.

In [1]:
%run grafo.ipynb
%run rotas.ipynb

N_MIN = 300
catalogo = pd.read_csv(PASTA_DADOS / "curados" / "catalogo_aisles_departamentos.csv")
demanda = pd.read_csv(PASTA_DADOS / "curados" / "frequencia_aisles.csv").set_index("aisle_id")["pedidos_prior"]
L0 = carregar_layout("L0")
secoes = sorted(L0)


def salvar_layout(nome, slot_da_secao):
    tabela = catalogo[["aisle_id", "aisle", "department_id", "department"]].copy()
    tabela[["corredor", "posicao_na_frente"]] = [loja.slots[slot_da_secao[s]] for s in tabela["aisle_id"]]
    tabela = tabela.sort_values(["corredor", "posicao_na_frente"])
    tabela.to_csv(ARQUIVOS_LAYOUT[nome], index=False)
    return tabela

## L1: afinidade

Trocar as seções $i$ e $j$ de lugar só muda as distâncias delas às demais, então a variação de $F$ é calculada sem refazer a soma inteira. A troca é aceita quando $F$ diminui, e a busca para quando nenhuma troca melhora, o que garante só um ótimo local.

In [2]:
indice = {secao: i for i, secao in enumerate(secoes)}
D = distancias[:loja.n_slots, :loja.n_slots]


def matriz_pesos(n_min):
    pesos = np.zeros((len(secoes), len(secoes)))
    for a, b, _, lift in arestas_g2(n_min).itertuples(index=False):
        pesos[indice[a], indice[b]] = pesos[indice[b], indice[a]] = lift - 1
    return pesos


def custo_F(posicao, pesos):
    return (pesos * D[np.ix_(posicao, posicao)]).sum() / 2


def busca_local(posicao, pesos):
    posicao = posicao.copy()
    melhorou = True
    while melhorou:
        melhorou = False
        for i in range(len(posicao)):
            for j in range(i + 1, len(posicao)):
                a, b = posicao[i], posicao[j]
                variacao = ((pesos[i] - pesos[j]) @ (D[b, posicao] - D[a, posicao])
                            + 2 * pesos[i, j] * D[a, b])
                if variacao < -1e-9:
                    posicao[i], posicao[j] = b, a
                    melhorou = True
    return posicao


pesos = matriz_pesos(N_MIN)
inicio = np.array([loja.indice_slot(*L0[s]) for s in secoes])
final = busca_local(inicio, pesos)
print(f"F(L0) = {custo_F(inicio, pesos):,.0f}  ->  F(L1) = {custo_F(final, pesos):,.0f}")
salvar_layout("L1", dict(zip(secoes, final)));

F(L0) = 152,898  ->  F(L1) = 95,371


## L2: popularidade

Empates de $\rho$ ficam com o menor número de corredor.

In [3]:
rho = distancias[loja.ENTRADA, :loja.n_slots] + distancias[:loja.n_slots, loja.CAIXA]
slots_por_custo = sorted(range(loja.n_slots), key=lambda p: (rho[p], loja.corredor[p]))
mais_pedidas = demanda.sort_values(ascending=False, kind="stable").index
salvar_layout("L2", dict(zip(mais_pedidas, slots_por_custo))).head()

,aisle_id,aisle,department_id,department,corredor,posicao_na_frente
23,24,fresh fruits,4,produce,1,1
15,16,fresh herbs,4,produce,1,2
44,45,candy chocolate,19,snacks,1,3
22,23,popcorn jerky,19,snacks,1,4
74,75,laundry,17,household,1,5


## Calibração de $n_{min}$

Para cada valor da grade, gera-se L1 e mede-se a distância média nos 1.000 pedidos de validação, com Held-Karp até 15 seções e vizinho mais próximo + 2-opt acima disso. Adota-se o valor de menor distância ou, entre os que ficam a menos de um erro-padrão dele (bootstrap pareado), o maior. A tabela é comparada com a da modelagem; a diferença de até 0,05 m vem da geometria, como em `testes.ipynb`. Leva cerca de 3 minutos.

In [4]:
GRADE = [30, 100, 300, 1000, 3215]
amostra = pd.read_csv(PASTA_DADOS / "amostras" / "amostra_validacao_1000.csv")
pedidos_validacao = [[int(secao) for secao in str(texto).split(";")] for texto in amostra["aisles"]]


def metros_na_validacao(posicao):
    slot_da_secao = dict(zip(secoes, posicao))
    metros = []
    for pedido in pedidos_validacao:
        slots = list(dict.fromkeys(slot_da_secao[secao] for secao in pedido))
        if usa_held_karp(slots):
            ordem = held_karp(slots, distancias, loja)
        else:
            ordem = dois_opt(vizinho_mais_proximo(slots, distancias, loja), distancias, loja)
        metros.append(custo_rota(ordem, distancias, loja))
    return np.array(metros)


def componentes(pesos):
    vistos, total = set(), 0
    for inicio_busca in range(len(secoes)):
        if inicio_busca in vistos:
            continue
        total += 1
        pilha = [inicio_busca]
        while pilha:
            atual = pilha.pop()
            if atual not in vistos:
                vistos.add(atual)
                pilha += list(np.flatnonzero(pesos[atual]))
    return total


metros = {}
linhas = []
for n_min in GRADE:
    pesos_n = matriz_pesos(n_min)
    metros[n_min] = metros_na_validacao(busca_local(inicio, pesos_n))
    linhas.append({"n_min": n_min, "arestas_G2": int(np.count_nonzero(np.triu(pesos_n))),
                   "componentes": componentes(pesos_n), "secoes_isoladas": int((~pesos_n.any(axis=1)).sum()),
                   "dist_media_validacao_m": metros[n_min].mean()})

calibracao = pd.DataFrame(linhas).set_index("n_min")
melhor = calibracao["dist_media_validacao_m"].idxmin()
sorteios = np.random.default_rng(42).integers(0, len(pedidos_validacao), size=(10000, len(pedidos_validacao)))
calibracao["diff_para_melhor_m"] = [(metros[n] - metros[melhor]).mean() for n in GRADE]
calibracao["EP_bootstrap_m"] = [(metros[n] - metros[melhor])[sorteios].mean(axis=1).std() for n in GRADE]
calibracao["dentro_de_1EP"] = calibracao["diff_para_melhor_m"] <= calibracao["EP_bootstrap_m"]
escolhido = calibracao.index[calibracao["dentro_de_1EP"]].max()

modelagem = pd.read_csv(PASTA_DADOS / "amostras" / "calibracao_n_min.csv").set_index("n_min")
colunas = ["arestas_G2", "componentes", "secoes_isoladas", "dentro_de_1EP"]
assert escolhido == N_MIN and (calibracao[colunas] == modelagem[colunas]).all().all()
assert (calibracao["dist_media_validacao_m"] - modelagem["dist_media_validacao_m"]).abs().max() <= 0.05
print(f"n_min escolhido: {escolhido} (igual ao da modelagem)")
calibracao.round(3)

n_min escolhido: 300 (igual ao da modelagem)


,arestas_G2,componentes,secoes_isoladas,dist_media_validacao_m,diff_para_melhor_m,EP_bootstrap_m,dentro_de_1EP
n_min,,,,,,,
30,8851,1,0,145.772,1.008,0.566,False
100,8582,1,0,148.812,4.048,0.511,False
300,7850,1,0,144.764,0.000,0.000,True
1000,6179,1,0,145.462,0.698,0.563,False
3215,4051,4,3,146.500,1.736,0.578,False
